**RAG chính sách (production):** pipeline đóng gói trong `rag/policy_pipeline.py` — hàm `run_policy_rag(query)`. Tool dùng trong agent: `tools/query_gym_policy.py` → `query_gym_policy`, node `policy_agent` khi router intent = `policy`.

In [33]:
import os, re
from typing import List, Dict, Any, Optional, TypedDict
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
try:
    from langchain_chroma import Chroma
except ImportError:
    from langchain_community.vectorstores import Chroma
from langchain_gradient import ChatGradient
from langchain_core.runnables import RunnableLambda


In [34]:
PERSIST_DIR = "../data/policy-terms-db"
COLLECTION_NAME = "gym-policy-terms"

EXPAND_N = 1
RETRIEVE_TOP_K = 6
RERANK_TOP_K = 4       

In [35]:
load_dotenv()

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5931.10it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
vectordb = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=PERSIST_DIR
)

/var/folders/1j/l08d1p2s6ls9235bnggg20rc0000gn/T/ipykernel_68885/1288059790.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


In [9]:
test = vectordb.similarity_search("Chính sách hoàn tiền")

print(test[0].page_content)

4. Gia hạn tự động: Một số gói có thể tự động gia hạn nếu Hội viên không thông báo huỷ 
trước thời hạn X ngày. Điều kiện này sẽ ghi rõ trong hợp đồng hoặc thông báo. 
5. Đóng băng (tạm ngưng): 
o Hội viên có thể yêu cầu đóng băng gói trong các trường hợp như: công tác dài 
ngày, thai kỳ, chấn thương có xác nhận y tế,… 
o Thời gian đóng băng tối đa và phí đóng băng (nếu có) sẽ nêu trong chính sách 
từng gói. 
o Yêu cầu đóng băng cần gửi trước thời điểm muốn áp dụng, không hồi tố. 
 
14. Chính sách huỷ, chấm dứt và hoàn tiền 
1. Huỷ trong thời gian "cooling-off" (nếu có quy định): 
o Trong X ngày đầu kể từ ngày kích hoạt gói, Hội viên có thể yêu cầu huỷ và hoàn 
một phần phí theo chính sách (trừ phí sử dụng thực tế, phí hành chính,…). 
2. Huỷ trước hạn đối với gói trả trước (3, 6, 12 tháng): 
o Thông thường không hoàn tiền trừ khi có quy định cụ thể hoặc trường hợp bất 
khả kháng (bệnh nặng, di cư,… kèm giấy tờ chứng minh). 
o Phòng Gym có thể áp dụng phí huỷ trước hạn. 
3. Chấm dứt từ p

In [10]:
llm = None
api_key = os.getenv("DIGITALOCEAN_INFERENCE_KEY") or os.getenv("GRADIENT_MODEL_ACCESS_KEY")
model_name = os.getenv("EDU_AGENT_MODEL", "openai-gpt-4o-mini")

if api_key:
    llm = ChatGradient(model=model_name, api_key=api_key)

from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="mps")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5761.63it/s]


In [13]:
from typing import Any


class RAGState(TypedDict):
    query: str
    expanded_queries: List[str]
    retrieved_docs: List[Dict[str, Any]]
    reranked_docs: List[Dict[str, Any]]
    answer: str

In [14]:
def expand_query_node(state: RAGState) -> RAGState:
    q = state["query"]
    prompt = f"""
    You are an expert in Customer Service. 
    Your task is expanding customer's questions in Vietnamese, 
    generate {EXPAND_N} queries from the question {q}, including the original question.
    Each query should be a standalone question and be written in Vietnamese.
    """
    resp = llm.invoke(prompt).content.splitlines()
    out = []
    for x in [q] + [s.strip() for s in resp]:
        if x and x not in out:
            out.append(x)
    state["expanded_queries"] = out
    return state

In [15]:
def retrieve_node(state: RAGState) -> RAGState:
    all_hits = {}

    for q in state["expanded_queries"]:
        hits = vectordb.similarity_search_with_relevance_scores(q, k=RETRIEVE_TOP_K)
        for doc, score in hits:
            key = doc.page_content.strip()
            if key not in all_hits or score > all_hits[key]["retrieval_score"]:
                all_hits[key] = {
                    "retrieval_score": score,
                    "doc": doc,
                    "matched_query": q,
                }

    state["retrieved_docs"] = sorted(all_hits.values(), key=lambda x: x["retrieval_score"], reverse=True)
    return state

In [16]:
def rerank_node(state: RAGState) -> RAGState:
    q = state["query"]
    cands = state["retrieved_docs"]
    pairs = [(q, c["doc"].page_content.strip()) for c in cands]
    scores = reranker.predict(pairs)
    ranked = []
    for c, s in zip(cands, scores):
        ranked.append({
            "doc": c["doc"],
            "matched_query": c["matched_query"],
            "reranking_score": s,
        })
    ranked.sort(key=lambda x: x["reranking_score"], reverse=True)
    state["reranked_docs"] = ranked
    return state

In [29]:
def answer_node(state: RAGState) -> RAGState:
    docs = state["reranked_docs"]

    if not docs:
        state["answer"] = "Không đủ dữ liệu để trả lời chắc chắn."
        return state

    ctx = "\n\n".join([
        f"[Context {i+1}]\n{d['doc'].page_content}"
        for i, d in enumerate(docs)
    ])

    if llm is None:
        state["answer"] = ctx[:1500]
        return state

    prompt = f"""
        Bạn là một Customer Service Agent chuyên trả lời câu hỏi về điều khoản sử dụng và chính sách.

        ## NHIỆM VỤ
        Trả lời câu hỏi của người dùng CHỈ dựa trên thông tin trong CONTEXT.

        ## QUY TẮC QUAN TRỌNG
        1. Chỉ sử dụng thông tin có trong CONTEXT. Không được suy đoán hoặc thêm kiến thức ngoài.
        2. Nếu CONTEXT không đủ để trả lời, phải trả lời:
        "Tài liệu không đề cập rõ thông tin này."
        3. Trả lời ngắn gọn, rõ ràng, đúng trọng tâm.
        4. Luôn trả lời bằng tiếng Việt.
        5. Khi có thông tin từ context, phải trích dẫn rõ:
        - Dùng format: (Điều khoản X.Y)
        - Có thể kết hợp nhiều context
        6. Không được fabricate điều khoản (ví dụ: "điều 6.1") nếu context không ghi rõ.

        ## FORMAT TRẢ LỜI
        - Trả lời trực tiếp câu hỏi
        - Sau mỗi ý quan trọng, thêm citation (Context X)

        ## USER QUESTION
        {state["query"]}

        ## CONTEXT
        {ctx}
        """

    state["answer"] = llm.invoke(prompt).content
    return state

In [30]:
rag_graph = (
    RunnableLambda(expand_query_node)
    | RunnableLambda(retrieve_node)
    | RunnableLambda(rerank_node)
    | RunnableLambda(answer_node)
)


In [31]:
def ask_rag(query: str):
    init = {
        "query": query,
        "expanded_queries": [],
        "retrieved_docs": [],
        "reranked_docs": [],
        "answer": "",
    }
    out = rag_graph.invoke(init)
    return out

out = ask_rag("Chính sách hoàn tiền của phòng gym ")
print(out["answer"])

Chính sách hoàn tiền của phòng gym được quy định như sau:

1. **Huỷ trong thời gian "cooling-off"**: Trong X ngày đầu kể từ ngày kích hoạt gói, Hội viên có thể yêu cầu huỷ và hoàn một phần phí theo chính sách (trừ phí sử dụng thực tế, phí hành chính,...) (Điều khoản 14.1).

2. **Huỷ trước hạn đối với gói trả trước**: Thông thường không hoàn tiền trừ khi có quy định cụ thể hoặc trường hợp bất khả kháng (bệnh nặng, di cư,… kèm giấy tờ chứng minh). Phòng gym có thể áp dụng phí huỷ trước hạn (Điều khoản 14.2).

3. **Chấm dứt từ phía Phòng Gym**: Nếu Phòng Gym chấm dứt hợp đồng mà không do lỗi Hội viên, sẽ hoàn lại phần phí tương ứng với thời gian còn lại chưa sử dụng (Điều khoản 14.3).

4. **Yêu cầu hoàn tiền**: Mọi yêu cầu hoàn tiền phải được gửi bằng văn bản (Email, form online hoặc phiếu tại quầy) và sẽ được xử lý trong thời hạn hợp lý tuỳ theo phương thức thanh toán ban đầu (Điều khoản 14.4). 

Nếu bạn cần thêm thông tin chi tiết, hãy cho tôi biết!
